In [1]:
from langchain_core.documents import Document

In [3]:
doc = Document(
    page_content = "This is the main text content used for RAG pipeline",
    metadata={
        "source": "example.txt",
        "pages": 1,
        "author": "Abhijit Kumbhar",
        "data_created": "2025-09-25"
    }
)
doc

Document(metadata={'source': 'example.txt', 'pages': 1, 'author': 'Abhijit Kumbhar', 'data_created': '2025-09-25'}, page_content='This is the main text content used for RAG pipeline')

In [16]:
from langchain_community.document_loaders import TextLoader, DirectoryLoader, PyPDFLoader, PyMuPDFLoader

In [9]:
loader = TextLoader("../data/how_car_engine_works.txt", encoding="utf-8")
loader.load()

[Document(metadata={'source': '../data/how_car_engine_works.txt'}, page_content="HOW A CAR ENGINE WORKS (THE BASICS)\n\nMost cars use an Internal Combustion Engine. This means fuel burns inside the engine to create power. \n\nThe heart of the engine is the CYLINDER. Inside the cylinder is a moving part called a PISTON. The piston moves up and down.\n\nMost car engines use a FOUR-STROKE CYCLE. This happens in four quick steps:\n\n1. INTAKE: The piston moves down. A valve opens to let fuel and air into the cylinder.\n2. COMPRESSION: The piston moves back up. It squeezes (compresses) the fuel and air mixture tightly.\n3. POWER: A spark plug creates a tiny spark. This ignites the fuel, causing a small explosion. The force pushes the piston down rapidly. This creates the engine's power.\n4. EXHAUST: The piston moves back up. Another valve opens to let the leftover burned gas out of the engine through the tailpipe.\n\nTURNING UP-AND-DOWN INTO FORWARD MOTION\nAs the pistons pump up and down, 

In [14]:
directory_loader = DirectoryLoader(
    "../data/text_files",
    loader_cls=TextLoader,
    loader_kwargs={'encoding': 'utf-8'},
    show_progress=False
)
directory_loader.load()

[Document(metadata={'source': '..\\data\\text_files\\car_engine_details.txt'}, page_content="HOW A CAR ENGINE WORKS: AN INTRODUCTORY GUIDE\n\nMost modern gasoline cars use an internal combustion engine that operates on a four-stroke cycle. This process converts the chemical energy in fuel into mechanical motion to drive the vehicle's wheels.\n\n=========================================\nTHE FOUR-STROKE CYCLE (THE OTTO CYCLE)\n=========================================\n\n1. INTAKE STROKE\n   - The piston moves downward inside the cylinder.\n   - The intake valve opens.\n   - A mixture of fresh air and fuel is drawn into the cylinder.\n\n2. COMPRESSION STROKE\n   - The intake and exhaust valves both close, sealing the cylinder.\n   - The piston moves back upward.\n   - This motion tightly squeezes (compresses) the air-fuel mixture, making the upcoming explosion significantly more powerful.\n\n3. POWER (COMBUSTION) STROKE\n   - The air-fuel mixture is fully compressed.\n   - The spark plu

In [20]:
pdf_loader = DirectoryLoader(
    "../data/pdf_files",
    glob="**/*.pdf",
    loader_cls=PyMuPDFLoader
)
pdf_loader.load()

[Document(metadata={'producer': 'Mac OS X 10.8.3 Quartz PDFContext', 'creator': 'Word', 'creationdate': "D:20130610081833Z00'00'", 'source': '..\\data\\pdf_files\\How-Car-Engine-Works.pdf', 'file_path': '..\\data\\pdf_files\\How-Car-Engine-Works.pdf', 'total_pages': 5, 'format': 'PDF 1.4', 'title': 'Microsoft Word - Car Engine.docx', 'author': 'Alberto Fernandez', 'subject': '', 'keywords': '', 'moddate': "D:20130610081833Z00'00'", 'trapped': '', 'modDate': "D:20130610081833Z00'00'", 'creationDate': "D:20130610081833Z00'00'", 'page': 0}, page_content='How\t\r \xa0Car\t\r \xa0Engines\t\r \xa0Work\t\r \xa0\n\t\r \xa0\n\t\r \xa0\n\t\r \xa0\n\t\r \xa0\n\t\r \xa0\n“A\t\r \xa0car\t\r \xa0engine\t\r \xa0is\t\r \xa0one\t\r \xa0of\t\r \xa0the\t\r \xa0most\t\r \xa0amazing\t\r \xa0machines\t\r \xa0we\t\r \xa0use\t\r \xa0\t\r \xa0\non\t\r \xa0a\t\r \xa0daily\t\r \xa0basis”\t\r \xa0\t\r \xa0Marshall\t\r \xa0Brian,\t\r \xa0creator\t\r \xa0of\t\r \xa0HowStufWorks.com\t\r \xa0\n\t\r \xa0\n\t\r \xa0\n\

In [22]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
# Pure numpy cosine similarity (no sklearn needed)
def _compute__compute_cosine_similarity(a, b):
    a_norm = np.linalg.norm(a, axis=1, keepdims=True)
    b_norm = np.linalg.norm(b, axis=1, keepdims=True)
    a_norm = np.where(a_norm == 0, 1e-10, a_norm)
    b_norm = np.where(b_norm == 0, 1e-10, b_norm)
    return np.dot(a / a_norm, (b / b_norm).T)


In [27]:
class EmbeddingManager:

    def __init__(self, model_name="all-MiniLM-L6-v2"):
        self.model_name = model_name
        self.model = None
        self._load_model()
    
    def _load_model(self):
        try:
            print("Loading model {}".format(self.model_name))
            self.model = SentenceTransformer(self.model_name)
            print("Model {} loaded successfully".format(self.model_name))
        except Exception as e:
            print("Error loading model: {}".format(self.model_name))
            raise
    
    def generate_embeddings(self, texts):
        if not self.model:
            raise ValueError("Model not loaded")
        print(f"Generating embeddings for {len(texts)}")
        embeddings = self.model.encode(texts, show_progress_bar=True)
        print(f"Generated embeddings with shape {embeddings.shape}")
        return embeddings

In [28]:
embedding_manager = EmbeddingManager()
embedding_manager

Loading model all-MiniLM-L6-v2


h:\AI\traditional_rag\.venv\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Abhijit\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
h:\AI\traditional_rag\.venv\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `hu

Model all-MiniLM-L6-v2 loaded successfully


In [45]:
from importlib.resources import path
class VectorStore:
    def __init__(self, collection_name="pdf_documents", persist_directory="../data/vecor_store"):
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()
    
    def _initialize_store(self):
        try:
            self.client = chromadb.PersistentClient(path=self.persist_directory)
            print("Initializing vectordb")
            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={"description": "PDF documents embeddings for RAG"}
            )
            print("Vectordb Initialized")
        except Exception as e:
            print("Error while initializing store")
            raise
    
    def add_documents(self, documents, embeddings):
        ids = []
        metadatas = []
        document_text = []
        embeddings_list = []

        for i, (doc, embeddings) in enumerate(zip(documents, embeddings)):
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)

            metadata = dict(doc.metadata)
            metadata['doc_index'] = i
            metadata['content_length'] = len(doc.page_content)
            metadatas.append(metadata)

            document_text.append(doc.page_content)
            embeddings_list.append(embeddings.tolist())
        
        try:
            self.collection.add(
                ids=ids,
                embeddings=embeddings_list,
                metadatas=metadatas,
                documents=document_text
            )
            print(f"Successfully added {len(documents)} to vector store")
        except Exception as e:
            print("Failed to add data to vectordb")
            raise



In [46]:
vector_store = VectorStore()
vector_store

Initializing vectordb
Vectordb Initialized


In [33]:
chunks = pdf_loader.load()

In [47]:
texts=[doc.page_content for doc in chunks]

# Generate the embeddings
embeddings = embedding_manager.generate_embeddings(texts)

# Store in the vecor database
vector_store.add_documents(chunks, embeddings)

Generating embeddings for 14


Batches: 100%|██████████| 1/1 [00:00<00:00,  1.58it/s]


Generated embeddings with shape (14, 384)
Successfully added 14 to vector store


In [48]:
embeddings

array([[-0.16784449,  0.06703836,  0.02977271, ...,  0.12542912,
         0.07761595,  0.04021515],
       [-0.03027885,  0.030384  ,  0.02011177, ...,  0.12543713,
         0.0792034 , -0.00116184],
       [-0.05053955,  0.03385167, -0.03058886, ...,  0.15751615,
         0.13700925, -0.04562545],
       ...,
       [-0.04193579, -0.01625229, -0.05683655, ...,  0.08629926,
         0.15513098, -0.05545658],
       [-0.06012842,  0.0039764 , -0.08341811, ...,  0.05981449,
         0.14107129, -0.06452756],
       [-0.11384989,  0.03147865,  0.0676023 , ...,  0.0579706 ,
        -0.04074186, -0.03612604]], shape=(14, 384), dtype=float32)

## 5. Retrieval Pipeline

In a Traditional RAG pipeline, the **Retriever** connects the user's natural language query to the stored vector knowledge:
1. **Query Embedding**: The user query is converted into a dense vector embedding using `embedding_manager`.
2. **Vector Similarity Search**: The vector store computes distances between the query vector and document vectors.
3. **Scoring & Filtering**: Distances are converted to normalized similarity scores, with optional score thresholding and metadata filters.
4. **Maximal Marginal Relevance (MMR)**: Balances query relevance and chunk diversity to eliminate redundant passages.
5. **Context Formatting**: Merges the retrieved chunks into clean context text with source citations ready for LLM generation.

In [ ]:
from typing import List, Tuple, Dict, Any, Optional
from langchain_core.documents import Document
import numpy as np
# Pure numpy cosine similarity (no sklearn needed)
def _compute__compute_cosine_similarity(a, b):
    a_norm = np.linalg.norm(a, axis=1, keepdims=True)
    b_norm = np.linalg.norm(b, axis=1, keepdims=True)
    a_norm = np.where(a_norm == 0, 1e-10, a_norm)
    b_norm = np.where(b_norm == 0, 1e-10, b_norm)
    return np.dot(a / a_norm, (b / b_norm).T)

class RAGRetriever:
    def __init__(self, vector_store, embedding_manager):
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager

    def retrieve_with_scores(
        self,
        query: str,
        top_k: int = 3,
        score_threshold: Optional[float] = None,
        filter_dict: Optional[Dict[str, Any]] = None
    ) -> List[Tuple[Document, float]]:
        """Retrieve top_k documents with normalized similarity scores (0 to 1)."""
        query_emb = self.embedding_manager.model.encode(query, show_progress_bar=False).tolist()
        
        query_kwargs = {
            "query_embeddings": [query_emb],
            "n_results": top_k,
            "include": ["documents", "metadatas", "distances"]
        }
        if filter_dict:
            query_kwargs["where"] = filter_dict

        results = self.vector_store.collection.query(**query_kwargs)
        docs = results.get("documents", [[]])[0]
        metadatas = results.get("metadatas", [[]])[0]
        distances = results.get("distances", [[]])[0]

        scored_results: List[Tuple[Document, float]] = []
        for text, meta, dist in zip(docs, metadatas, distances):
            # Normalized similarity score from L2 distance (1.0 = exact match)
            similarity = 1.0 / (1.0 + float(dist))
            if score_threshold is not None and similarity < score_threshold:
                continue
            doc = Document(page_content=text, metadata=meta or {})
            scored_results.append((doc, similarity))

        return scored_results

    def retrieve(
        self,
        query: str,
        top_k: int = 3,
        score_threshold: Optional[float] = None,
        filter_dict: Optional[Dict[str, Any]] = None
    ) -> List[Document]:
        """Retrieve top_k Document objects matching the query."""
        scored = self.retrieve_with_scores(query, top_k, score_threshold, filter_dict)
        return [doc for doc, _ in scored]

    def retrieve_mmr(
        self,
        query: str,
        top_k: int = 3,
        fetch_k: int = 10,
        lambda_mult: float = 0.5,
        filter_dict: Optional[Dict[str, Any]] = None
    ) -> List[Document]:
        """Maximal Marginal Relevance (MMR) retrieval for diverse, non-redundant chunks."""
        query_emb = self.embedding_manager.model.encode(query, show_progress_bar=False)
        total_docs = self.vector_store.collection.count()
        fetch_k = min(fetch_k, total_docs)
        if fetch_k <= 0:
            return []

        query_kwargs = {
            "query_embeddings": [query_emb.tolist()],
            "n_results": fetch_k,
            "include": ["documents", "metadatas", "embeddings"]
        }
        if filter_dict:
            query_kwargs["where"] = filter_dict

        results = self.vector_store.collection.query(**query_kwargs)
        docs = results.get("documents", [[]])[0]
        metadatas = results.get("metadatas", [[]])[0]
        candidate_embeddings = results.get("embeddings", [[]])[0]

        if not docs or not candidate_embeddings:
            return []

        cand_embs = np.array(candidate_embeddings)
        q_emb_2d = np.array(query_emb).reshape(1, -1)
        query_sims = _compute_cosine_similarity(cand_embs, q_emb_2d).reshape(-1)

        selected_indices = []
        remaining_indices = list(range(len(docs)))

        for _ in range(min(top_k, len(docs))):
            best_mmr_score = -float("inf")
            best_idx = None

            for idx in remaining_indices:
                relevance = query_sims[idx]
                if not selected_indices:
                    diversity_penalty = 0.0
                else:
                    selected_embs = cand_embs[selected_indices]
                    sims_to_selected = _compute_cosine_similarity(cand_embs[idx].reshape(1, -1), selected_embs).reshape(-1)
                    diversity_penalty = float(np.max(sims_to_selected))

                mmr_score = lambda_mult * relevance - (1.0 - lambda_mult) * diversity_penalty
                if mmr_score > best_mmr_score:
                    best_mmr_score = mmr_score
                    best_idx = idx

            if best_idx is not None:
                selected_indices.append(best_idx)
                remaining_indices.remove(best_idx)

        return [Document(page_content=docs[i], metadata=metadatas[i] or {}) for i in selected_indices]

    def format_context(self, documents: List[Document]) -> str:
        """Format retrieved chunks into a prompt context string for the LLM."""
        if not documents:
            return "No relevant documents found."
        context_blocks = []
        for i, doc in enumerate(documents, 1):
            source = doc.metadata.get("source", "Unknown")
            page = doc.metadata.get("page", None)
            page_str = f" | Page: {page}" if page is not None else ""
            block = f"[Document {i}]\nSource: {source}{page_str}\nContent:\n{doc.page_content.strip()}"
            context_blocks.append(block)
        return "\n\n" + ("=" * 50) + "\n\n" + "\n\n".join(context_blocks) + "\n\n" + ("=" * 50)


In [ ]:
# Initialize the retriever with the existing vector_store and embedding_manager
retriever = RAGRetriever(vector_store=vector_store, embedding_manager=embedding_manager)
print("Retriever successfully initialized!")


In [ ]:
# 1. Similarity Retrieval with Scores
query = "How does the four-stroke cycle work?"
results = retriever.retrieve_with_scores(query, top_k=2)

print(f"Query: {query}\n")
for i, (doc, score) in enumerate(results, 1):
    print(f"=== Rank {i} (Similarity Score: {score:.4f}) ===")
    print(f"Source: {doc.metadata.get('source')} | Page: {doc.metadata.get('page')}")
    print(f"Content:\n{doc.page_content[:300].strip()}...\n")


In [ ]:
# 2. Diversity Retrieval with MMR (Maximal Marginal Relevance)
query_mmr = "What are the main components and parts of an engine?"
mmr_results = retriever.retrieve_mmr(query_mmr, top_k=3, lambda_mult=0.6)

print(f"MMR Query: {query_mmr}\n")
for i, doc in enumerate(mmr_results, 1):
    print(f"=== MMR Selected Chunk {i} (Page {doc.metadata.get('page')}) ===")
    print(f"{doc.page_content[:250].strip()}...\n")


In [ ]:
# 3. Formatted Context for LLM Generation
prompt_context = retriever.format_context([doc for doc, _ in results])
print("--- LLM Prompt Ready Context ---")
print(prompt_context)
